<a href="https://colab.research.google.com/github/sinbbang/L-IntelligentSystem/blob/main/%EC%A7%80%EB%8A%A5%EC%8B%9C%EC%8A%A4%ED%85%9C_07-8_Agentic_RAG_Action_Space_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 실습 8 — Agentic RAG Action Space

**목표:** 같은 Harness에서 질문마다 다음 행동을 동적으로 선택하는 과정을 관찰합니다.

- Action Space: `search_vector_db`, `search_keyword`, `open_document`, `calculate`, `ask_user`, `finish`
- **질문 1:** 벡터 검색 → 문서 열기 → 계산 → 종료
- **질문 2:** 사용자 추가 질문 → 키워드 검색 → 벡터 검색 → 문서 열기 → 종료
- **중요:** 단순히 경로가 다르다고 Agent인 것은 아닙니다. 규칙으로 경로를 정한 모드와 **LLM이 상태를 보고 행동을 결정하는 모드**를 구분합니다.

기본 실습은 API 키 없이 실행됩니다. Gemini 연동은 선택 사항입니다.

## 1. 데이터와 행동 공간 준비

In [1]:
ACTIONS = ["search_vector_db", "search_keyword", "open_document", "calculate", "ask_user", "finish"]

# 학습용 소형 문서 저장소(실제 벡터 데이터베이스는 사용하지 않습니다.)
DOCUMENTS = {
    "grade": {"title": "평가 계획", "text": "지능시스템 과목의 과제 비중은 20%, 중간고사는 30%, 기말고사는 30%, 출석은 20%입니다."},
    "submission": {"title": "과제 제출", "text": "과제는 Google Colab 노트북(.ipynb) 파일을 LMS에 제출합니다."},
    "ai": {"title": "AI 사용 정책", "text": "AI 도구를 사용해도 되지만 사용 사실과 출처를 명시해야 합니다."},
}
print("사용 가능한 행동:", ACTIONS)

사용 가능한 행동: ['search_vector_db', 'search_keyword', 'open_document', 'calculate', 'ask_user', 'finish']


## 2. 도구 만들기 — 입력과 출력 형식을 직접 확인

In [2]:
def search_vector_db(query):
    """의미 검색을 흉내 내는 교육용 도구입니다. 실제 임베딩 검색은 아닙니다."""
    q = query.lower()
    if any(w in q for w in ["과제", "평가", "비율", "점수", "30%"]):
        return [{"doc_id": "grade", "title": DOCUMENTS["grade"]["title"]}]
    if any(w in q for w in ["제출", "노트북", "colab"]):
        return [{"doc_id": "submission", "title": DOCUMENTS["submission"]["title"]}]
    return [{"doc_id": "ai", "title": DOCUMENTS["ai"]["title"]}]

def search_keyword(query):
    """문서 제목 또는 본문에 검색 단어가 포함되는지 확인합니다."""
    words = query.lower().split()
    return [{"doc_id": key, "title": doc["title"]}
            for key, doc in DOCUMENTS.items()
            if any(word in (doc["title"] + " " + doc["text"]).lower() for word in words)]

def open_document(doc_id):
    """검색 결과에 나온 문서 ID로 원문을 엽니다."""
    doc = DOCUMENTS.get(doc_id)
    return {"doc_id": doc_id, **doc} if doc else {"error": "unknown_doc_id"}

def calculate(a, b, operation="add"):
    """정해진 두 숫자의 사칙연산 일부를 안전하게 처리합니다."""
    if operation == "add": return a + b
    if operation == "multiply": return a * b
    raise ValueError("지원하지 않는 계산입니다")

def ask_user(question):
    """교육용 모의 사용자 응답. 실제 사용자와 대화하지는 않습니다."""
    return {"question": question, "answer": "AI 사용 정책도 함께 알려주세요."}

TOOLS = {"search_vector_db": search_vector_db, "search_keyword": search_keyword,
         "open_document": open_document, "calculate": calculate, "ask_user": ask_user}

## 3. Harness — 선택된 행동을 검증·실행하고 관찰 결과 저장

In [3]:
def run_agent(question, policy, max_steps=8):
    """Policy가 행동을 고르고, Harness는 실행과 안전 제한을 담당합니다."""
    state = {"question": question, "observations": []}
    log = []
    for step in range(1, max_steps + 1):
        action = policy(state)                 # 다음 행동 선택
        if not isinstance(action, dict) or not isinstance(action.get("tool"), str):
            return None, log, "invalid_action"
        name = action["tool"]
        args = action.get("args", {})
        if name not in ACTIONS or not isinstance(args, dict):
            return None, log, "invalid_action"
        if name == "finish":
            answer = action.get("answer", args.get("answer"))
            if not isinstance(answer, str) or not answer.strip():
                return None, log, "invalid_answer"
            log.append({"step": step, "tool": name, "answer": answer})
            return answer, log, "completed"
        try:
            result = TOOLS[name](**args)      # 실제 함수 실행
        except (TypeError, ValueError) as exc:
            return None, log + [{"step": step, "tool": name, "error": str(exc)}], "tool_error"
        entry = {"step": step, "tool": name, "args": args, "result": result}
        log.append(entry)
        state["observations"].append(entry)  # 다음 결정에 결과 반영
    return None, log, "max_steps"

def show_run(label, outcome):
    answer, log, status = outcome
    print(f"\n[{label}] 상태: {status} | 답변: {answer}")
    print("실행 경로:", " → ".join(row["tool"] for row in log))
    for item in log:
        print(" ", item)


## 4. 규칙 기반 정책으로 두 경로 재현하기 (LLM 미사용)

아래 `demo_policy`는 결과를 예측 가능하게 보여주는 **모의 정책**입니다. 따라서 이 코드만으로 실제 LLM Agent를 구현했다고 주장하지 않습니다.

In [4]:
def demo_policy(state):
    question = state["question"]
    history = state["observations"]
    n = len(history)
    # 질문 1: 검색 → 문서 열기 → 계산 → 종료
    if "합계" in question:
        steps = [
            {"tool": "search_vector_db", "args": {"query": question}},
            {"tool": "open_document", "args": {"doc_id": "grade"}},
            {"tool": "calculate", "args": {"a": 20, "b": 30, "operation": "add"}},
            {"tool": "finish", "answer": "과제 20%와 중간고사 30%의 합계는 50%입니다."},
        ]
    # 질문 2: 추가 질문 → 키워드 검색 → 벡터 검색 → 문서 열기 → 종료
    else:
        steps = [
            {"tool": "ask_user", "args": {"question": "어떤 AI 관련 사항이 궁금한가요?"}},
            {"tool": "search_keyword", "args": {"query": "AI 사용"}},
            {"tool": "search_vector_db", "args": {"query": "AI 출처 사용 정책"}},
            {"tool": "open_document", "args": {"doc_id": "ai"}},
            {"tool": "finish", "answer": "AI 도구를 사용할 수 있지만 사용 사실과 출처를 명시해야 합니다."},
        ]
    return steps[min(n, len(steps)-1)]

q1 = "지능시스템의 과제와 중간고사 평가 비율 합계를 알려줘"
q2 = "AI를 활용할 때 지켜야 할 정책이 궁금해"
result1 = run_agent(q1, demo_policy)
result2 = run_agent(q2, demo_policy)
show_run("질문 1", result1)
show_run("질문 2", result2)


[질문 1] 상태: completed | 답변: 과제 20%와 중간고사 30%의 합계는 50%입니다.
실행 경로: search_vector_db → open_document → calculate → finish
  {'step': 1, 'tool': 'search_vector_db', 'args': {'query': '지능시스템의 과제와 중간고사 평가 비율 합계를 알려줘'}, 'result': [{'doc_id': 'grade', 'title': '평가 계획'}]}
  {'step': 2, 'tool': 'open_document', 'args': {'doc_id': 'grade'}, 'result': {'doc_id': 'grade', 'title': '평가 계획', 'text': '지능시스템 과목의 과제 비중은 20%, 중간고사는 30%, 기말고사는 30%, 출석은 20%입니다.'}}
  {'step': 3, 'tool': 'calculate', 'args': {'a': 20, 'b': 30, 'operation': 'add'}, 'result': 50}
  {'step': 4, 'tool': 'finish', 'answer': '과제 20%와 중간고사 30%의 합계는 50%입니다.'}

[질문 2] 상태: completed | 답변: AI 도구를 사용할 수 있지만 사용 사실과 출처를 명시해야 합니다.
실행 경로: ask_user → search_keyword → search_vector_db → open_document → finish
  {'step': 1, 'tool': 'ask_user', 'args': {'question': '어떤 AI 관련 사항이 궁금한가요?'}, 'result': {'question': '어떤 AI 관련 사항이 궁금한가요?', 'answer': 'AI 사용 정책도 함께 알려주세요.'}}
  {'step': 2, 'tool': 'search_keyword', 'args': {'query': 'AI 사용'}, 'result': 

## 5. 두 경로 비교 및 검증

In [5]:
path1 = [x["tool"] for x in result1[1]]
path2 = [x["tool"] for x in result2[1]]
assert path1 == ["search_vector_db", "open_document", "calculate", "finish"]
assert path2 == ["ask_user", "search_keyword", "search_vector_db", "open_document", "finish"]
assert result1[2] == result2[2] == "completed"
assert run_agent(q1, demo_policy, max_steps=2)[2] == "max_steps"
print("두 질문의 서로 다른 경로와 최대 단계 제한 검증 성공!")

두 질문의 서로 다른 경로와 최대 단계 제한 검증 성공!


## 6. (선택) Gemini를 정책 모델로 연결하기

실제 모델이 `state`의 관찰 결과를 읽고 다음 행동을 선택하도록 합니다. 앞의 규칙 기반 정책과 달리 결과 경로가 실행마다 다를 수 있습니다.

1. Colab 왼쪽 **비밀정보(열쇠 아이콘)**에 `GEMINI_API_KEY` 등록
2. 다음 설치 및 정책 정의 셀 실행
3. API 호출은 사용량 제한·비용이 발생할 수 있으므로 한 질문만 시험 권장

**참고:** 모의 `ask_user`는 자동으로 고정 답변을 반환합니다. 실제 서비스라면 사용자 입력 대기와 재개 기능이 필요합니다.

In [7]:
%pip -q install -U google-genai

In [8]:
import json, os
from google import genai
from google.genai import types

def get_api_key():
    try:
        from google.colab import userdata
        return userdata.get("GEMINI_API_KEY")
    except (ImportError, KeyError):
        return os.getenv("GEMINI_API_KEY")

API_KEY = get_api_key()
# 자신의 Gemini 계정에서 사용 가능한 모델로 변경해도 됩니다.
MODEL_ID = "gemini-3.5-flash-lite"

ALLOWED = set(ACTIONS)

def gemini_policy(state):
    if not API_KEY:
        raise RuntimeError("GEMINI_API_KEY를 Colab 비밀정보에 등록하세요")
    client = genai.Client(api_key=API_KEY)
    prompt = f"""당신은 교육용 RAG Agent 정책입니다.
질문 및 도구 실행 기록을 보고 다음 행동 하나를 고르세요.
사용 가능한 행동: {ACTIONS}
인자 형식:
search_vector_db: {{"query": "문자열"}}
search_keyword: {{"query": "문자열"}}
open_document: {{"doc_id": "grade 또는 submission 또는 ai"}}
calculate: {{"a": 20, "b": 30, "operation": "add"}}
ask_user: {{"question": "문자열"}}
finish: {{"answer": "근거에 기반한 최종 답변"}}
절대 검색 결과를 지어내지 마세요. 모르면 모른다고 말하세요.
finish 전에 가능하면 검색 문서 원문을 확인하세요.
반드시 tool과 args를 포함한 JSON 객체 하나만 반환하세요.
상태: {json.dumps(state, ensure_ascii=False)}"""
    response = client.models.generate_content(
        model=MODEL_ID, contents=prompt,
        config=types.GenerateContentConfig(response_mime_type="application/json", temperature=0)
    )
    if not response.text:
        return {"tool": "finish", "answer": "모델 응답이 비어 있어 종료했습니다."}
    action = json.loads(response.text)
    # LLM 출력과 Harness 사이의 인터페이스 계약을 정규화합니다.
    if "tool" not in action and isinstance(action.get("action"), str):
        action["tool"] = action.pop("action")
    if action.get("tool") == "finish":
        action["answer"] = action.get("answer") or action.get("args", {}).get("answer", "")
    return action

print("Gemini 정책 준비:", "API 키 확인됨" if API_KEY else "API 키 없음 — 선택 실습 건너뛰기")

Gemini 정책 준비: API 키 확인됨


In [9]:
# API 키가 없으면 이 셀은 자동으로 건너뜁니다.
if API_KEY:
    try:
        llm_outcome = run_agent("지능시스템 과제 비율은 얼마인가요?", gemini_policy, max_steps=5)
        show_run("실제 Gemini 정책", llm_outcome)
    except Exception as exc:
        print("Gemini 호출 실패:", type(exc).__name__, str(exc)[:300])
else:
    print("API 키 없음: 앞의 모의 정책 실습은 정상 실행되었습니다.")


[실제 Gemini 정책] 상태: completed | 답변: 지능시스템 과목의 과제 비중은 20%입니다.
실행 경로: search_vector_db → open_document → finish
  {'step': 1, 'tool': 'search_vector_db', 'args': {'query': '지능시스템 과제 비율'}, 'result': [{'doc_id': 'grade', 'title': '평가 계획'}]}
  {'step': 2, 'tool': 'open_document', 'args': {'doc_id': 'grade'}, 'result': {'doc_id': 'grade', 'title': '평가 계획', 'text': '지능시스템 과목의 과제 비중은 20%, 중간고사는 30%, 기말고사는 30%, 출석은 20%입니다.'}}
  {'step': 3, 'tool': 'finish', 'answer': '지능시스템 과목의 과제 비중은 20%입니다.'}


## 7. 학생 활동 및 정리

1. 질문 1에서 `calculate`를 빼면 최종 답변의 신뢰성을 어떻게 검증할 수 있을까요?
2. `max_steps=2`로 제한하면 어떤 종료 상태가 기록되나요?
3. `demo_policy()`는 상태에 따라 경로를 고르지만 사전에 작성된 규칙입니다. 실제 LLM 정책과 무엇이 다를까요?
4. `open_document`에 존재하지 않는 문서 ID를 입력하면 도구가 어떤 결과를 반환하나요?

**핵심:** 넓은 Action Space와 런타임 의사결정이 Agentic RAG를 설명하는 중요한 요소입니다. 그러나 **행동 이름이 많다는 사실만으로는 Agent가 되지 않으며**, 정책이 실제로 관찰 결과와 목표를 바탕으로 다음 행동을 선택하는지가 중요합니다.

**실습 범위:** 본 노트북의 `search_vector_db`는 실제 벡터 DB가 아닌 교육용 모사 함수입니다. 실습 4~7과 마찬가지로 먼저 Harness와 제어 구조를 이해하는 데 초점을 맞춥니다.